**RAPIDS cuML is a GPU-accelerated machine learning library with a scikit-learn-like API that dramatically speeds up training on large datasets.**

# RAPIDS cuML SVC Baseline

In [7]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import cudf
from cuml.svm import SVC
from sklearn.model_selection import KFold

import os

# ------------------- IMPORT SRC ------------------------------------
# src is the parent folder of notebooks, so we need to add it to sys.path to import config and utils
import sys
notebook_dir = os.getcwd() 

# Parent folder of src
project_root = os.path.abspath(os.path.join(notebook_dir, "..")) 
sys.path.append(project_root)

print("sys.path contains:", sys.path[-1])

from src import config, utils  
# -------------------------------------------------------

sys.path contains: /home/ismail/Documents/projects/ml-projects/icr


In [8]:
# -------------------------------
# Load Data
# -------------------------------
X_train = pd.read_csv('../data/X_train_encoded.csv')
X_test  = pd.read_csv('../data/X_test_encoded.csv')
y_train = pd.read_csv('../data/y_train.csv')
# test_ids = X_test['PassengerId']
# X_test.drop(columns=['PassengerId'], inplace=True)

# Flatten target if needed
# Map target to numeric
target = config.Target

y_train_numeric = y_train[target]


num_classes = len(np.unique(y_train_numeric))
print(f"Number of classes: {num_classes}")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Number of classes: 2
X_train shape: (617, 57)
X_test shape: (5, 58)


In [16]:
from cuml.svm import SVC as cuSVC
from sklearn.model_selection import KFold
import cudf
import pandas as pd
import numpy as np

BAGS = 20
FOLDS = 11

# Ensure y_train is 1D Series
if isinstance(y_train, pd.DataFrame):
    y_train = y_train.squeeze()  # Convert single-column DataFrame to Series

oof = np.zeros(len(X_train))
models = {}

for bag in range(BAGS):
    print('#'*25)
    print('### Bag', bag+1)
    print('#'*25)

    models[bag] = []
    skf = KFold(n_splits=FOLDS, shuffle=True, random_state=bag)

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X=X_train, y=y_train)):
        print(f'=> Fold {fold+1}, ', end='')

        # TRAIN/VALID split
        X_train_fold = X_train.iloc[train_idx].copy()
        y_train_fold = y_train.iloc[train_idx].copy()
        X_valid_fold = X_train.iloc[valid_idx].copy()
        y_valid_fold = y_train.iloc[valid_idx].copy()

        # DOWNSAMPLE NEGATIVE CLASS
        remove_idx = y_train_fold[y_train_fold == 0].sample(
            frac=0.7, random_state=bag*BAGS + fold
        ).index
        X_train_fold = X_train_fold.drop(index=remove_idx)
        y_train_fold = y_train_fold.drop(index=remove_idx)

        # Move data to GPU
        X_train_gpu = cudf.DataFrame.from_pandas(X_train_fold)
        y_train_gpu = cudf.Series(y_train_fold)
        X_valid_gpu = cudf.DataFrame.from_pandas(X_valid_fold)

        # Initialize cuML SVC with probability
        clf = cuSVC(C=5, probability=True, random_state=42)
        clf.fit(X_train_gpu, y_train_gpu)

        # Predict probabilities
        proba_df = clf.predict_proba(X_valid_gpu)  # cuDF DataFrame
        oof[valid_idx] += proba_df.iloc[:,1].to_pandas().values / BAGS

        # Save model
        models[bag].append(clf)

    print()

#########################
### Bag 1
#########################
=> Fold 1, [2026-03-08 01:19:14.389] [CUML] [warning] Random state is currently ignored by probabilistic SVC


ValueError: Column must have no nulls.